# TransformerLens: Interactive Tour

Same content as [`../01_notebooks/00_transformerlens_tour.ipynb`](../01_notebooks/00_transformerlens_tour.ipynb), restructured for
typing practice: every worked example lives in the markdown above its cell
as reference, and the code cell itself is blank. Type it yourself, then
run it. Pure library mechanics, no interpretability theory yet (that starts
in [`../01_notebooks/01_transformer_primer.ipynb`](../01_notebooks/01_transformer_primer.ipynb)).


```python
import torch
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # we're only doing inference in this whole track, never training
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
```


In [ ]:
# type it yourself, then run


## Loading a model

`HookedTransformer.from_pretrained(name)` downloads (and caches locally
after the first run) a model's weights from Hugging Face and wraps them in
TransformerLens's instrumented implementation. It works for a long list of
open models (GPT-2 family, Pythia, Llama, Mistral, Qwen, ...). You're not
limited to GPT-2, that's just the small, fast one we're using for exercises.


```python
model = HookedTransformer.from_pretrained("gpt2", device=device)
```


In [ ]:
# type it yourself, then run


## Tokenization: text <-> tokens

Four methods cover almost everything you need:

- `model.to_tokens(text)`: string -> tensor of token ids, shape `[batch, pos]`
- `model.to_str_tokens(text)`: string -> list of the individual token *strings* (useful for labeling plots)
- `model.to_string(tokens)`: tokens -> string, the inverse of `to_tokens`
- `model.to_single_token(text)`: string -> a single token id, and it'll error loudly if `text` isn't exactly one token (a good sanity check when you assume something is one token and want to be told if you're wrong)

By default `to_tokens` prepends a beginning-of-sequence token
(`<|endoftext|>` for GPT-2). Pass `prepend_bos=False` if you don't want
that.


```python
tokens = model.to_tokens("hello world")
print("with BOS:   ", tokens)
print("without BOS:", model.to_tokens("hello world", prepend_bos=False))
print("as strings: ", model.to_str_tokens("hello world"))
print("round trip: ", model.to_string(tokens))
```


In [ ]:
# type it yourself, then run


## Running the model

Two ways to run a forward pass, and the difference matters:

- `model(tokens)`: a plain forward pass. Fast, returns only the final
  logits. Use this when you just want the model's output and don't care
  about internals.
- `model.run_with_cache(tokens)`: runs the same forward pass but also
  records every intermediate activation. Slower and more memory, but this is
  the one mech interp actually runs on almost every call, since the whole
  point is inspecting internals.


```python
logits = model(tokens)
print("plain forward, logits shape:", logits.shape)  # [batch, pos, d_vocab]

logits, cache = model.run_with_cache(tokens)
print("run_with_cache, same logits shape:", logits.shape, "  plus a cache with", len(cache), "entries")
```


In [ ]:
# type it yourself, then run


## Reading the cache

`cache` is a dict-like `ActivationCache`. Two equivalent ways to index it:

- Full hook name: `cache["blocks.0.attn.hook_pattern"]`
- Shorthand tuple: `cache["pattern", 0]`. TransformerLens infers the
  `blocks.{layer}.attn.hook_` prefix for you. Handy, but only works for
  activation names it recognizes this way; when in doubt, use the full name.

`utils.get_act_name("pattern", 0)` builds the full name string
programmatically. Prefer this over hand-typing hook name strings in real
code, since it can't typo.


```python
full_name = "blocks.0.attn.hook_pattern"
shorthand = cache["pattern", 0]
via_name = cache[full_name]
via_util = cache[utils.get_act_name("pattern", 0)]

print(torch.equal(shorthand, via_name), torch.equal(via_name, via_util))
print(shorthand.shape)  # [batch, head, dest_pos, src_pos]
```


In [ ]:
# type it yourself, then run


## Generating text

`model.generate` does autoregressive sampling for you. Pass a string or
tokens, get a string or tokens back. `do_sample=False` makes it
deterministic (greedy decoding), which is what you want for reproducible
debugging.


```python
completion = model.generate("The capital of France is", max_new_tokens=5, do_sample=False, verbose=False)
print(completion)
```


In [ ]:
# type it yourself, then run


## `utils.test_prompt`: the fastest sanity-check tool

Give it a prompt and an expected answer; it tells you the answer's rank and
probability, plus what the model actually would have said instead. This is
the tool to reach for first whenever you're about to build a whole
experiment around "the model predicts X here." Check that claim in one
line before writing ten lines of analysis on top of it.


```python
utils.test_prompt(
    "The Eiffel Tower is in the city of",
    " Paris",
    model,
    prepend_space_to_answer=False,
)
```


In [ ]:
# type it yourself, then run


## Hooks, at the mechanism level

A **hook** is a named tap point inserted into the forward pass. You've only
*read* from them so far (via `run_with_cache`); you can also intercept and
*edit* the value flowing through one, which is how causal experiments like
activation patching work later on.

`model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` runs one forward pass
with `fn` inserted at the hook called `name`. `fn` receives the activation
tensor and a `hook` object (which has `.name`, so one function can be reused
across many hook points), and must return the (possibly modified) tensor.


```python
def zero_out(activation, hook):
    print(f"  intercepted {hook.name}, shape {tuple(activation.shape)}")
    return torch.zeros_like(activation)

# ablate layer 0's attention output entirely and see the prediction degrade
clean_logits = model(tokens)
ablated_logits = model.run_with_hooks(tokens, fwd_hooks=[("blocks.0.attn.hook_z", zero_out)])

print("same logits?", torch.allclose(clean_logits, ablated_logits))
```


In [ ]:
# type it yourself, then run


## `model.cfg`: architecture constants

`model.cfg` holds every architecture constant you'll use as a loop bound or
tensor shape: `n_layers`, `n_heads`, `d_model`, `d_vocab`, and more. Printing
it once is worth doing so you know what's available.


```python
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_model, model.cfg.d_vocab)
```


In [ ]:
# type it yourself, then run

## `model.W_E` and `model.W_U`: raw embedding matrices

`model.W_E` (shape `[d_vocab, d_model]`) is the token embedding matrix.
`model.W_U` (shape `[d_model, d_vocab]`) is the unembedding matrix that
turns a residual-stream vector into logits. Slicing a single column out of
`W_U` gives you the direction in residual-stream space that increases a
particular token's logit, the building block of direct logit attribution.


```python
print(model.W_E.shape, model.W_U.shape)
paris_token = model.to_single_token(" Paris")
paris_direction = model.W_U[:, paris_token]   # [d_model]
print(paris_direction.shape)
```


In [ ]:
# type it yourself, then run

## Reading the residual stream layer by layer

`cache.accumulated_resid(layer=-1, incl_mid=True, return_labels=True)`
returns the running sum of the residual stream *as of* each layer (and,
with `incl_mid=True`, each sub-block too), stacked into one tensor of shape
`[n_checkpoints, batch, pos, d_model]`, along with a label for each
checkpoint.

Before projecting any of those vectors through `W_U`, you need to apply the
model's final LayerNorm to them. LayerNorm's scale is computed per-position
from the *final* residual stream, not from each intermediate one, so you
can't just call `.layer_norm(...)` on them yourself.
`cache.apply_ln_to_stack(resid_stack, layer=-1)` does this correctly.


```python
resid_stack, labels = cache.accumulated_resid(layer=-1, incl_mid=True, return_labels=True)
print(resid_stack.shape, len(labels))  # [n_checkpoints, batch, pos, d_model]

scaled_stack = cache.apply_ln_to_stack(resid_stack, layer=-1)
print(scaled_stack.shape)  # same shape, now safe to project through W_U
```


In [ ]:
# type it yourself, then run

## Per-head contributions to the residual stream

`cache.stack_head_results(layer=-1, return_labels=True)` stacks every
individual attention head's contribution to the residual stream (before
they get summed together) into one tensor of shape
`[n_layers * n_heads, batch, pos, d_model]`, with a label per head. This is
the mechanic behind direct logit attribution *per head*: project each head's
row through a logit direction to see how much that one head pushed the
answer.


```python
per_head_resid, head_labels = cache.stack_head_results(layer=-1, return_labels=True)
print(per_head_resid.shape, len(head_labels))
```


In [ ]:
# type it yourself, then run

## Parameterizing a hook with `functools.partial`

`run_with_hooks` always calls your hook as `fn(activation, hook)`, exactly
two arguments. To pass in something extra (which position to patch, a value
to patch *in*), bind it ahead of time with `functools.partial`, same as you
practiced in the torch notebook. Inside the hook, mutating the activation
with slice assignment and returning it is the exact mechanic activation
patching runs on.


```python
from functools import partial

def patch_position(activation, hook, position, replacement):
    activation[:, position, :] = replacement
    return activation

clean_logits, clean_cache = model.run_with_cache(tokens)
replacement = clean_cache["blocks.0.hook_resid_pre"][:, 0, :]
hook_fn = partial(patch_position, position=0, replacement=replacement)
patched_logits = model.run_with_hooks(tokens, fwd_hooks=[("blocks.0.hook_resid_pre", hook_fn)])
print(torch.allclose(clean_logits, patched_logits))
```


In [ ]:
# type it yourself, then run

## `get_act_name` beyond `"pattern"`

The same shorthand works for other activation names: `"resid_pre"`,
`"resid_post"`, `"mlp_out"`, `"q"`, `"k"`, `"v"`, and more. The pattern is
always `blocks.<layer>.<attn|mlp|(none)>.hook_<name>`; `get_act_name` knows
the right prefix for each name so you don't have to memorize it.


```python
print(utils.get_act_name("resid_pre", 3))
print(utils.get_act_name("mlp_out", 3))
print(cache[utils.get_act_name("resid_pre", 3)].shape)
```


In [ ]:
# type it yourself, then run

## Plotting convention: diverging vs. sequential color scales

Every plot of a *signed* quantity (attribution scores that can be positive
or negative, patching effects) in the notebooks ahead uses
`color_continuous_scale="RdBu", color_continuous_midpoint=0`, so zero renders
white and the sign is visually obvious. Plots of a strictly *non-negative*
quantity (attention weights, induction scores) use `"Blues"` instead. Always
`.cpu()` a tensor before handing it to `plotly`.


```python
import plotly.express as px

signed = torch.randn(4, 6)
px.imshow(signed.cpu(), color_continuous_scale="RdBu", color_continuous_midpoint=0).show()

nonneg = torch.rand(4, 6)
px.imshow(nonneg.cpu(), color_continuous_scale="Blues").show()
```


In [ ]:
# type it yourself, then run

## Checkpoint: write a few lines yourself

Using `cache` from above, compute `resid_norms`: a 1D tensor of shape
`[n_layers + 1]` holding the L2 norm (over `d_model`) of the accumulated
residual stream at the **last token position**, for each layer checkpoint
(embedding included, mid-layer points excluded). Use
`cache.accumulated_resid(layer=-1, incl_mid=False, return_labels=True)` and
`.norm(dim=-1)`.

In [ ]:
resid_norms = None  # YOUR CODE HERE

In [ ]:
# self-check
assert resid_norms is not None, "fill in `resid_norms` above"
_stack, _labels = cache.accumulated_resid(layer=-1, incl_mid=False, return_labels=True)
_expected = _stack[:, 0, -1, :].norm(dim=-1)
assert resid_norms.shape == _expected.shape, f"expected shape {_expected.shape}, got {resid_norms.shape}"
assert torch.allclose(resid_norms, _expected, atol=1e-4), f"got {resid_norms}"
print("Checkpoint passed")

## Cheat sheet

| Want to... | Call |
|---|---|
| Load a model | `HookedTransformer.from_pretrained(name, device=device)` |
| Text -> tokens | `model.to_tokens(text)` |
| Tokens -> readable strings | `model.to_str_tokens(text)` |
| Tokens -> text | `model.to_string(tokens)` |
| Text -> exactly one token id | `model.to_single_token(text)` |
| Fast forward pass, logits only | `model(tokens)` |
| Forward pass + every activation | `model.run_with_cache(tokens)` |
| Read one activation | `cache[utils.get_act_name(name, layer)]` or `cache[name, layer]` |
| Generate text | `model.generate(text, max_new_tokens=n, do_sample=False)` |
| Quick "does the model predict X" check | `utils.test_prompt(prompt, answer, model)` |
| Run with an activation intercepted/edited | `model.run_with_hooks(tokens, fwd_hooks=[(name, fn)])` |
| Model architecture constants | `model.cfg.d_model`, `.n_layers`, `.n_heads`, etc. |
| Raw embed / unembed matrices | `model.W_E` `[d_vocab, d_model]`, `model.W_U` `[d_model, d_vocab]` |
| Residual stream accumulated up to each layer | `cache.accumulated_resid(layer=-1, incl_mid=, return_labels=True)` |
| Correctly apply final LayerNorm to an intermediate resid stack | `cache.apply_ln_to_stack(resid_stack, layer=-1)` |
| Per-head contribution to the residual stream | `cache.stack_head_results(layer=-1, return_labels=True)` |
| Parameterize a hook beyond `(activation, hook)` | `functools.partial(hook_fn, extra=val)` |
| Edit an activation in place inside a hook | `activation[:, position, :] = replacement; return activation` |
| Signed-quantity plot | `px.imshow(t.cpu(), color_continuous_scale="RdBu", color_continuous_midpoint=0)` |
| Non-negative-quantity plot | `px.imshow(t.cpu(), color_continuous_scale="Blues")` |

## Next

[`../01_notebooks/01_transformer_primer.ipynb`](../01_notebooks/01_transformer_primer.ipynb): same tools, now used to actually
explain the residual stream, attention, and MLP conceptually.